# QAI Practical Exam – All 6 Practicals
**Qiskit / Jupyter Notebook preparation**

## Environment setup
The supplied lab manual uses the older Qiskit 0.44-era APIs. Run the installation cell if needed. Restart the kernel after installation.

> These examples use a local simulator and do not require an IBM Quantum account.


In [ ]:
%pip install qiskit==0.44.2 qiskit-aer==0.12.2 qiskit-ignis==0.7.1 matplotlib numpy

In [ ]:
import qiskit
import qiskit_aer
print("Qiskit:", qiskit.__version__)
print("Aer:", qiskit_aer.__version__)

# Practical 1 – 16 Qubit Random Number Generator\n\nPractical 1 of 6.

## Aim
Implement a 16-qubit Quantum Random Number Generator using Hadamard gates and measurement.

In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import Aer
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt

# Create 16 qubits and 16 classical bits
circuit = QuantumCircuit(16, 16)

# Put every qubit into an equal superposition
circuit.h(range(16))

# Measure every qubit
circuit.measure(range(16), range(16))

# Display the circuit
print(circuit)
circuit.draw(output="mpl")

In [ ]:
# Run one measurement (one 16-bit random result)
simulator = Aer.get_backend("qasm_simulator")
job = simulator.run(circuit, shots=1)
result = job.result()
counts = result.get_counts(circuit)

binary_number = next(iter(counts))
random_number = int(binary_number, 2)

print("Measurement counts:", counts)
print("Random number (binary):", binary_number)
print("Random number (decimal):", random_number)

### Theory
A qubit can be in a superposition of |0> and |1>. The Hadamard gate applied to |0> creates an equal superposition, so measurement gives 0 or 1 with probability 50% in an ideal system. Measuring 16 qubits produces a 16-bit result. Its unsigned decimal range is 0 to 65,535.

### Algorithm
1. Import Qiskit libraries.
2. Create 16 qubits and 16 classical bits.
3. Apply Hadamard gates to all qubits.
4. Measure all qubits.
5. Run the circuit on the QASM simulator.
6. Read the binary result and convert it to decimal.

### Conclusion
Thus, a 16-qubit quantum random number generator was implemented using Qiskit.

# Practical 2 – Tackle Noise with Error Mitigation\n\nPractical 2 of 6.

## Aim
Demonstrate measurement noise and reduce its effect using measurement calibration and mitigation.

**Compatibility:** This experiment uses Qiskit Ignis, which belongs to the older Qiskit 0.44-era environment.

In [ ]:
from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer
from qiskit_aer.noise import NoiseModel, ReadoutError
from qiskit.ignis.mitigation.measurement import (
    CompleteMeasFitter,
    complete_meas_cal
)
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt

In [ ]:
# Prepare a three-qubit GHZ state: (|000> + |111>) / sqrt(2)
qc = QuantumCircuit(3, 3)
qc.h(0)
qc.cx(0, 1)
qc.cx(0, 2)
qc.measure(range(3), range(3))

print(qc)
qc.draw(output="mpl")

In [ ]:
# Define a readout noise model.
# Each measured bit has a 10% chance of being reported incorrectly.
noise_model = NoiseModel()
readout_error = ReadoutError([
    [0.90, 0.10],
    [0.10, 0.90]
])
noise_model.add_all_qubit_readout_error(readout_error)

backend = Aer.get_backend("qasm_simulator")
transpiled_qc = transpile(
    qc, backend, basis_gates=noise_model.basis_gates
)

job = backend.run(
    transpiled_qc, shots=10000, noise_model=noise_model
)
counts = job.result().get_counts()
print("Original noisy counts:", counts)

In [ ]:
# Build calibration circuits for all 3 measured qubits
cal_circuits, state_labels = complete_meas_cal(qubit_list=[0, 1, 2])
cal_circuits = transpile(
    cal_circuits, backend, basis_gates=noise_model.basis_gates
)

cal_job = backend.run(
    cal_circuits, shots=10000, noise_model=noise_model
)
cal_results = cal_job.result()

# Fit the calibration data and mitigate the noisy counts
meas_fitter = CompleteMeasFitter(cal_results, state_labels)
mitigated_counts = meas_fitter.filter.apply(counts)

print("Mitigated counts:", mitigated_counts)

In [ ]:
plot_histogram(
    [counts, mitigated_counts],
    legend=["Original noisy", "Mitigated"]
)
plt.show()

### Theory
Quantum noise is an unwanted disturbance caused by imperfect operations, measurement errors, and interaction with the environment. Measurement error mitigation estimates readout errors using known calibration states, then uses that calibration to improve the measured distribution.

This code demonstrates **measurement error mitigation**, not full quantum error correction. Error correction encodes information across multiple qubits and uses syndrome measurements to detect and correct errors.

### Algorithm
1. Create a three-qubit GHZ circuit.
2. Add a readout-noise model.
3. Run the circuit and collect noisy counts.
4. Generate known-state calibration circuits.
5. Run the calibration circuits with the same noise.
6. Build a measurement fitter.
7. Apply the fitter to the original counts.
8. Compare original and mitigated histograms.

### Conclusion
Thus, measurement noise was demonstrated and its effect was reduced using measurement calibration and mitigation.

**Note:** Mitigated counts can be floating-point values and may not sum to an exact integer. Results depend on shots and calibration quality.

# Practical 3 – Quantum Coin Flipping\n\nPractical 3 of 6.

## Aim
Implement quantum coin flipping using a Hadamard gate and repeated measurement.

In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import Aer
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt

# Create one qubit and one classical bit
qc = QuantumCircuit(1, 1)

# Create an equal superposition
qc.h(0)

# Measure the qubit
qc.measure(0, 0)

print(qc)
qc.draw(output="mpl")

In [ ]:
# Perform 1000 coin flips
num_shots = 1000
simulator = Aer.get_backend("qasm_simulator")
job = simulator.run(qc, shots=num_shots)
counts = job.result().get_counts()

print("Coin flip results:", counts)

for outcome, count in sorted(counts.items()):
    label = "Head" if outcome == "0" else "Tail"
    percentage = count / num_shots * 100
    print(f"{label}: {count} ({percentage:.2f}%)")

In [ ]:
plot_histogram(counts, legend=["Coin flip"])
plt.show()

### Theory
A Hadamard gate applied to |0> creates an equal superposition of |0> and |1>. When measured, the qubit gives either result with 50% probability in an ideal system. We map 0 to Head and 1 to Tail. Repeating the circuit many times gives an approximately balanced distribution; it need not be exactly 50/50.

### Algorithm
1. Create one qubit and one classical bit.
2. Apply a Hadamard gate.
3. Measure the qubit.
4. Run the circuit for 1000 shots.
5. Count the 0 and 1 outcomes.
6. Display percentages and a histogram.

### Conclusion
Thus, quantum coin flipping was implemented using Qiskit.

# Practical 4 – Quantum Teleportation\n\nPractical 4 of 6.

## Aim
Implement quantum teleportation using three qubits, entanglement, classical measurement results, and conditional corrections.

This notebook teleports the known example state |1>. The same circuit structure can teleport a general unknown state.

In [ ]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit_aer import Aer
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt

# q[0]: source qubit; q[1]: Alice's Bell-pair qubit; q[2]: Bob's qubit
q = QuantumRegister(3, "q")
alice0 = ClassicalRegister(1, "alice0")
alice1 = ClassicalRegister(1, "alice1")
bob_result = ClassicalRegister(1, "bob")
circuit = QuantumCircuit(q, alice0, alice1, bob_result)

In [ ]:
# Prepare the source state |1>
circuit.x(q[0])
circuit.barrier()

# Create a Bell pair shared by Alice and Bob
circuit.h(q[1])
circuit.cx(q[1], q[2])
circuit.barrier()

# Alice performs the Bell-basis operations
circuit.cx(q[0], q[1])
circuit.h(q[0])
circuit.barrier()

# Alice measures her two qubits
circuit.measure(q[0], alice0[0])
circuit.measure(q[1], alice1[0])
circuit.barrier()

# Bob applies corrections based on Alice's classical results
# If alice1 = 1, apply X. If alice0 = 1, apply Z.
circuit.x(q[2]).c_if(alice1, 1)
circuit.z(q[2]).c_if(alice0, 1)

# Measure Bob's final qubit
circuit.measure(q[2], bob_result[0])

print(circuit)
circuit.draw(output="mpl")

In [ ]:
# Run many shots to observe all possible Alice outcomes
simulator = Aer.get_backend("qasm_simulator")
job = simulator.run(circuit, shots=1000)
counts = job.result().get_counts()

print("Teleportation results:", counts)
print("The Bob register should always be 1 for the input state |1>.")
plot_histogram(counts)
plt.show()

### Theory
Quantum teleportation transfers a quantum state from a sender (Alice) to a receiver (Bob). It uses three qubits: the source qubit, Alice's half of an entangled Bell pair, and Bob's half. Alice performs CNOT and Hadamard operations, measures her two qubits, and communicates two classical bits to Bob. Bob uses those bits to apply conditional X and Z corrections.

The state is transferred, not copied. The original state is destroyed by Alice's measurement, consistent with the no-cloning theorem.

### Algorithm
1. Prepare the source qubit.
2. Create an entangled Bell pair between Alice and Bob.
3. Apply CNOT from the source to Alice's qubit.
4. Apply Hadamard to the source qubit.
5. Measure Alice's two qubits.
6. Send the two classical measurement bits to Bob.
7. Apply conditional X and Z corrections.
8. Measure Bob's qubit.

### Correction rule
For classical results (source measurement, Alice's Bell-pair measurement): 00 = no correction; 01 = X; 10 = Z; 11 = both X and Z. The code uses separate classical registers and applies the equivalent conditional operations.

### Conclusion
Thus, quantum teleportation was implemented using entanglement and classical communication.

# Practical 5 – Randomized Benchmarking Demonstration\n\nPractical 5 of 6.

## Aim
Demonstrate randomized benchmarking ideas by applying randomly generated unitary circuits followed by their inverse and measuring return to the initial state.

**Important:** This is a simplified inverse-circuit demonstration, not the full standard Clifford-group randomized benchmarking protocol.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit_aer import Aer
import matplotlib.pyplot as plt

In [ ]:
def generate_random_circuit(num_qubits, depth):
    circuit = QuantumCircuit(num_qubits)

    for _ in range(depth):
        for qubit in range(num_qubits):
            circuit.rx(np.random.uniform(0, 2 * np.pi), qubit)
            circuit.ry(np.random.uniform(0, 2 * np.pi), qubit)
            circuit.rz(np.random.uniform(0, 2 * np.pi), qubit)

        for qubit in range(num_qubits - 1):
            circuit.cz(qubit, qubit + 1)

    return circuit

In [ ]:
def randomized_benchmarking_demo(
    num_qubits=2,
    depths=(1, 2, 3, 4, 5),
    num_sequences=20,
    shots=100
):
    backend = Aer.get_backend("qasm_simulator")
    success_rates = []

    for depth in depths:
        successful_shots = 0
        total_shots = num_sequences * shots

        for _ in range(num_sequences):
            random_circuit = generate_random_circuit(num_qubits, depth)
            inverse_circuit = random_circuit.inverse()

            test_circuit = QuantumCircuit(num_qubits, num_qubits)
            test_circuit.compose(random_circuit, inplace=True)
            test_circuit.compose(inverse_circuit, inplace=True)
            test_circuit.measure(range(num_qubits), range(num_qubits))

            result = backend.run(test_circuit, shots=shots).result()
            counts = result.get_counts()

            # The ideal initial state was |00...0>
            successful_shots += counts.get("0" * num_qubits, 0)

        success_rates.append(successful_shots / total_shots)

    return list(depths), success_rates

In [ ]:
depths, results = randomized_benchmarking_demo()

print("Randomized benchmarking demonstration:")
for depth, rate in zip(depths, results):
    print(f"Depth {depth}: success rate = {rate * 100:.2f}%")

In [ ]:
plt.plot(depths, results, marker="o")
plt.xlabel("Random circuit depth")
plt.ylabel("Return-to-zero success rate")
plt.title("Simplified Randomized Benchmarking Demonstration")
plt.ylim(0, 1.05)
plt.grid(True)
plt.show()

### Theory
Randomized benchmarking is used to estimate the performance of quantum gates. In the standard protocol, random Clifford gate sequences are applied, followed by a recovery operation. The probability of returning to the expected state is measured for different sequence lengths. The decay of this probability provides information about average gate errors.

This practical code is a simplified demonstration: it generates random RX, RY, RZ and CZ operations, appends the exact inverse, and measures whether the system returns to |00...0>. On an ideal simulator the success rate is approximately 100%. To observe errors, a noise model or real hardware is needed.

### Algorithm
1. Import Qiskit and NumPy.
2. Generate random quantum circuits of selected depths.
3. Create the inverse of each random circuit.
4. Append the original circuit and its inverse.
5. Measure all qubits.
6. Repeat for multiple random sequences.
7. Calculate the return-to-zero success rate.
8. Plot success rate against circuit depth.

### Conclusion
Thus, a simplified randomized benchmarking demonstration was implemented using Qiskit.

# Practical 6 – Five-Qubit Quantum Fourier Transform\n\nPractical 6 of 6.

## Aim
Implement a five-qubit Quantum Fourier Transform (QFT) using Qiskit and observe the measurement distribution.

In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import Aer
from qiskit.circuit.library import QFT
from qiskit.visualization import plot_histogram
import matplotlib.pyplot as plt

# Create five qubits and five classical bits
qc = QuantumCircuit(5, 5)

In [ ]:
# Prepare the computational basis input |10101>
# q[4], q[2], q[0] are set to 1
qc.x(4)
qc.x(2)
qc.x(0)

# Append the five-qubit QFT
qft = QFT(
    num_qubits=5,
    approximation_degree=0,
    do_swaps=True,
    inverse=False
)
qc.append(qft, range(5))

# Measure all qubits
qc.measure(range(5), range(5))

print(qc)
qc.draw(output="mpl")

In [ ]:
# Execute on the QASM simulator
backend = Aer.get_backend("qasm_simulator")
job = backend.run(qc, shots=1000)
result = job.result()
counts = result.get_counts()

print("QFT output counts:")
print(counts)

plot_histogram(counts, title="Five-Qubit QFT Output")
plt.show()

### Theory
The Quantum Fourier Transform converts amplitudes of a quantum state into the Fourier basis. It is used in important quantum algorithms, including phase estimation and Shor's algorithm.

For N = 2^n, the transform is:

QFT|x> = (1/sqrt(N)) sum from y=0 to N-1 of exp(2*pi*i*x*y/N)|y>

In this experiment n = 5, so N = 2^5 = 32. The input is a computational basis state. An ideal QFT of a basis state has equal measurement probabilities across all 32 output bit strings; the phase information is not visible from computational-basis measurement counts alone.

### Algorithm
1. Import Qiskit libraries.
2. Create five qubits and five classical bits.
3. Prepare the input state using X gates.
4. Create a five-qubit QFT circuit.
5. Append QFT to the circuit.
6. Measure all qubits.
7. Execute for 1000 shots.
8. Display counts and histogram.

### Conclusion
Thus, a five-qubit Quantum Fourier Transform was implemented using Qiskit.